# Analyzing Images

In this notebook, we send an image to an OpenAI model and ask it to identify the visible ingredients in a fridge.

We also ask for rough amounts, so the response includes estimates like milliliters, grams, or pieces.

Docs:
- [Images and vision](https://developers.openai.com/api/docs/guides/images-vision)
- [File inputs](https://developers.openai.com/api/docs/guides/file-inputs)

## Setup

In [ ]:
import base64
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Image, display
from openai import OpenAI
from pydantic import BaseModel

load_dotenv(override=True)
client = OpenAI()

## Load the image

The fridge image is stored in the local `files` folder. We display it first so we know what we are sending to the model.

In [ ]:
FRIDGE_IMAGE_PATH = Path("files/fridge.png")
display(Image(filename=str(FRIDGE_IMAGE_PATH)))

## Defining the model

[OpenAI Models](https://developers.openai.com/api/docs/models)

[GPT-4o](https://developers.openai.com/api/docs/models/gpt-4o)

[GPT-5.5](https://developers.openai.com/api/docs/models/gpt-5.5)

In [ ]:
MODEL = "gpt-5.5"

## Define the data we want back

A Pydantic model gives the response a clear shape. Each ingredient has a name, a rough amount, a unit, and optional notes for uncertainty.

In [ ]:
class Ingredient(BaseModel):
    name: str
    amount: float | None
    unit: str | None
    notes: str | None


class FridgeAnalysis(BaseModel):
    ingredients: list[Ingredient]

## Define instructions

In [ ]:
INSTRUCTIONS = (
    "Analyze the fridge image and identify visible ingredients. "
    "Return one item per ingredient or package. "
    "Estimate rough amounts. "
    "Use grams for solid loose ingredients, milliliters for liquids, "
    "and pieces for countable items or packages. "
    "Use null for amount and unit if no reasonable estimate is possible. "
    "Use short notes for uncertainty, packaging, or partially hidden items."
)

## Option 1 - Web URL

If an image already lives online, send it with an `image_url`. The URL must be public so the API can fetch it.

In [ ]:
FRIDGE_IMAGE_URL = (
    "https://raw.githubusercontent.com/LukasLechnerDev/"
    "AI-Engineering-Foundations-Labs/main/"
    "3-interacting-with-llm-apis/files/fridge.png"
)

response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_image",
                    "image_url": FRIDGE_IMAGE_URL,
                },
            ],
        }
    ],
    text_format=FridgeAnalysis,
)

fridge_analysis = response.output_parsed

if fridge_analysis is not None:
    print(fridge_analysis.model_dump_json(indent=2))

## Option 2 - Send a base64 encoded image

For a local one-off image, base64 is the simplest option. We send the image bytes directly in the request.

In [ ]:
fridge_base64 = base64.b64encode(FRIDGE_IMAGE_PATH.read_bytes()).decode("utf-8")

response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{fridge_base64}",
                },
            ],
        }
    ],
    text_format=FridgeAnalysis,
)

fridge_analysis = response.output_parsed

if fridge_analysis is not None:
    print(fridge_analysis.model_dump_json(indent=2))

## Option 3 - File upload

If you want to reuse the same image in multiple requests, upload it once and then reference the returned file id.

In [ ]:
print("Start uploading the image to the OpenAI API...")

with FRIDGE_IMAGE_PATH.open("rb") as image_file:
    uploaded_image = client.files.create(
        file=image_file,
        purpose="vision",
    )

print("Upload completed. File ID:", uploaded_image.id)

response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_image",
                    "file_id": uploaded_image.id,
                },
            ],
        }
    ],
    text_format=FridgeAnalysis,
)

fridge_analysis = response.output_parsed

if fridge_analysis is not None:
    print(fridge_analysis.model_dump_json(indent=2))

## Things worth remembering

- Use a web URL when the image is already public.
- Use base64 when you have a local image and only need one request.
- Use file upload when you want to reuse the same image across multiple requests.
- Ask for rough amounts explicitly. Otherwise the model may only return ingredient names.